# DermaVision - Phase 3: FL (IID) - MobileNetV2 (High Accuracy)

**Configuration**: Fine-Tuning with **Frozen BatchNormalization**.
This strategy achieved ~73% accuracy in the baseline phase.

---

In [ ]:
# Ensure dependencies are installed via requirements.txt or pip
# !pip install -q flwr[simulation] tensorflow

In [1]:
import numpy as np
import tensorflow as tf
import os
import shutil

BACKUP_DIR = 'backup/DermaVision'

# Load Data
if os.path.exists('X_train.npy'):
    print("Loading data from local storage...")
elif os.path.exists(os.path.join(BACKUP_DIR, 'X_train.npy')):
    print(f"Loading data from backup ({BACKUP_DIR})...")
    for file in ['X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy', 'X_test.npy', 'y_test.npy', 'label_classes.npy']:
        shutil.copy(os.path.join(BACKUP_DIR, file), '.')
else:
    print("ERROR: Data not found! Please run 01_DataSetup.ipynb first.")

if os.path.exists('X_train.npy'):
    X_train = np.load('X_train.npy')
    y_train = np.load('y_train.npy')
    X_val = np.load('X_val.npy')
    y_val = np.load('y_val.npy')
    X_test = np.load('X_test.npy')
    y_test = np.load('y_test.npy')
    label_classes = np.load('label_classes.npy')

Loading data from local storage...


In [4]:
# Split IID
if 'X_train' in locals():
    NUM_CLIENTS = 10
    indices = np.random.permutation(len(X_train))
    X_shuffled = X_train[indices]
    y_shuffled = y_train[indices]
    X_clients = np.array_split(X_shuffled, NUM_CLIENTS)
    y_clients = np.array_split(y_shuffled, NUM_CLIENTS)

In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, Rescaling, RandomFlip, RandomRotation
from tensorflow.keras.optimizers import Adam

def create_model():
    """Create MobileNetV2 with Fine-Tuning enabled but BN Frozen."""
    inputs = Input(shape=(128, 128, 3))
    
    # Data Augmentation
    x = RandomFlip("horizontal_and_vertical")(inputs)
    x = RandomRotation(0.2)(x)
    x = Rescaling(scale=2.0, offset=-1.0)(x)
    
    # Base
    base_model = MobileNetV2(
        input_tensor=x,
        include_top=False,
        weights='imagenet'
    )
    
    # Fine-Tuning: Unfreeze weights, but FREEZE BN stats
    base_model.trainable = True
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    
    # Head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(3, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
import gc

NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32

if 'X_clients' in locals():
    print(f"Starting FL (Fine-Tuned + Frozen BN)...")

    global_model = create_model()
    # Low learning rate for fine-tuning stability
    global_model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
    global_weights = global_model.get_weights()

    round_accuracies = []

    for round_num in range(1, NUM_ROUNDS + 1):
        print(f"\nRound {round_num}/{NUM_ROUNDS}")
        client_weights = []
        client_sizes = []
        
        for i in range(NUM_CLIENTS):
            tf.keras.backend.clear_session()
            client_model = create_model()
            client_model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
            client_model.set_weights(global_weights)
            
            client_model.fit(X_clients[i], y_clients[i], epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
            
            client_weights.append(client_model.get_weights())
            client_sizes.append(len(X_clients[i]))
            del client_model
            gc.collect()
        
        # FedAvg
        total_size = sum(client_sizes)
        new_weights = []
        for layer_idx in range(len(client_weights[0])):
            layer_avg = np.zeros_like(client_weights[0][layer_idx])
            for client_idx in range(NUM_CLIENTS):
                weight = client_sizes[client_idx] / total_size
                layer_avg += weight * client_weights[client_idx][layer_idx]
            new_weights.append(layer_avg)
        
        tf.keras.backend.clear_session()
        global_model = create_model()
        global_model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
        
        global_weights = new_weights
        global_model.set_weights(global_weights)
        
        loss, acc = global_model.evaluate(X_val, y_val, verbose=0)
        round_accuracies.append(acc)
        print(f"  -> Val Acc: {acc*100:.2f}%")

    print("FL Complete")

Starting FL (Fine-Tuned + Frozen BN)...


C:\Users\Sev\AppData\Local\Temp\ipykernel_6580\3893019789.py:16: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(



Round 1/50

  -> Val Acc: 38.67%

Round 2/50
  -> Val Acc: 39.56%

Round 3/50
  -> Val Acc: 47.11%

Round 4/50
  -> Val Acc: 51.33%

Round 5/50
  -> Val Acc: 51.11%

Round 6/50
  -> Val Acc: 53.56%

Round 7/50
  -> Val Acc: 54.89%

Round 8/50
  -> Val Acc: 55.11%

Round 9/50
  -> Val Acc: 54.44%

Round 10/50
  -> Val Acc: 53.78%

Round 11/50
  -> Val Acc: 55.33%

Round 12/50
  -> Val Acc: 55.11%

Round 13/50
  -> Val Acc: 56.89%

Round 14/50
  -> Val Acc: 56.67%

Round 15/50
  -> Val Acc: 56.67%

Round 16/50
  -> Val Acc: 58.00%

Round 17/50
  -> Val Acc: 57.56%

Round 18/50
  -> Val Acc: 58.67%

Round 19/50
  -> Val Acc: 57.78%

Round 20/50
  -> Val Acc: 58.00%

Round 21/50
  -> Val Acc: 58.44%

Round 22/50
  -> Val Acc: 58.89%

Round 23/50


In [ ]:
if 'global_model' in locals():
    loss, acc = global_model.evaluate(X_test, y_test, verbose=0)
    print(f"Test Accuracy: {acc*100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

if 'global_model' in locals():
    y_pred = global_model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    cm = confusion_matrix(y_true_classes, y_pred_classes)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=label_classes, yticklabels=label_classes)
    plt.title('FL (IID) Model - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig('fl_iid_confusion_matrix.png', dpi=100)
    plt.show()

    print("Saved fl_iid_confusion_matrix.png")

In [ ]:
if 'y_true_classes' in locals() and 'y_pred_classes' in locals():
    print("\nClassification Report:")
    print(classification_report(y_true_classes, y_pred_classes, target_names=label_classes))

In [ ]:
# Sample Predictions
if 'X_test' in locals():
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    indices = np.random.choice(len(X_test), 10, replace=False)

    for i, idx in enumerate(indices):
        row = i // 5
        col = i % 5
        axes[row, col].imshow(X_test[idx])
        axes[row, col].axis('off')
        true_label = label_classes[y_true_classes[idx]]
        pred_label = label_classes[y_pred_classes[idx]]
        color = 'green' if true_label == pred_label else 'red'
        axes[row, col].set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=10)

    plt.suptitle('FL (IID) - Sample Predictions', fontsize=14)
    plt.tight_layout()
    plt.savefig('fl_iid_predictions.png', dpi=100)
    plt.show()

In [ ]:
# Save
if 'global_model' in locals():
    global_model.save('fl_iid_model.keras')
    with open('fl_iid_results.txt', 'w') as f:
        f.write(f"Test Accuracy: {acc*100:.2f}%")

    # Backup to Backup Directory
    os.makedirs(BACKUP_DIR, exist_ok=True)
    shutil.copy('fl_iid_model.keras', BACKUP_DIR)
    for file in glob.glob('fl_iid_*.png'):
        shutil.copy(file, BACKUP_DIR)
    shutil.copy('fl_iid_results.txt', BACKUP_DIR)
    
    print(f"Results backed up to {BACKUP_DIR}")